# 📘 Assignment 1 — RAG and Prompt Engineering

> **Objective:** Build a complete Retrieval-Augmented Generation (RAG) pipeline using LangChain, OpenAI, and ChromaDB to answer questions from a PDF article.

This notebook walks through the following stages:

| # | Stage | Description |
|---|-------|-------------|
| 1 | **Environment Setup** | Import libraries and configure API keys |
| 2 | **Baseline Prompting** | Query GPT-4.1 without any document context |
| 3 | **Prompt Engineering** | Add a system prompt to guide model behaviour |
| 4 | **Document Ingestion** | Load a PDF, count tokens, and split into chunks |
| 5 | **Embedding & Vector Store** | Generate embeddings and store them in ChromaDB |
| 6 | **RAG Pipeline** | Retrieve relevant context and generate grounded answers |
| 7 | **Edge-Case Testing** | Validate that out-of-scope questions return "no context found" |
| 8 | **Inference** | Insights and recommendations for the business problems |

---

## Stage 1 — Environment Setup

### 1.1 Import Libraries 

This cell performs two tasks:


1. **Library imports** — We load all required packages:
   - `PDFPlumberLoader` — extracts text from PDFs with superior layout/table handling.
   - `AutoTokenizer` (HuggingFace) — BPE tokenizer for token counting and chunk sizing.
   - `RecursiveCharacterTextSplitter` — splits documents into overlapping token-based chunks.
   - `OpenAIEmbeddings` & `Chroma` — generate vector embeddings and index them in ChromaDB.
   - `OpenAI` — direct client for chat completions.

In [1]:
# ──────────────────────────────────────────────────────────────────────────────
# Keep notebook output clean 
# ──────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"                                 
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"                          
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"                             

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)                      
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)                   
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)       

# ──────────────────────────────────────────────────────────────────────────────
# Import core libraries
# ──────────────────────────────────────────────────────────────────────────────
import json                                                                     # Read/write JSON data
import requests  # type: ignore                                                 # Make HTTP requests (e.g., API calls)

# ──────────────────────────────────────────────────────────────────────────────
# Import PDF loader and OpenAI client
# ──────────────────────────────────────────────────────────────────────────────
from langchain_community.document_loaders import PDFPlumberLoader              # Extract text from PDFs with superior table/layout handling
from openai import OpenAI                                                       # OpenAI Python client for chat completions

# ──────────────────────────────────────────────────────────────────────────────
# Tokenizer and data utilities
# ──────────────────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer                                          # HuggingFace tokenizer for token counting and chunk sizing
import pandas as pd                                                             # Tabular data manipulation

# ──────────────────────────────────────────────────────────────────────────────
# LangChain components — splitting, embedding, vector store
# ──────────────────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter             # Split text into overlapping chunks
from langchain_openai import OpenAIEmbeddings                                   # Generate vector embeddings via OpenAI API
from langchain_chroma import Chroma                                             # ChromaDB vector store integration
from langchain_openai import ChatOpenAI                                         # LangChain wrapper for OpenAI chat models
from datasets import Dataset                                                    # Structure evaluation datasets in tabular form

### 1.2 Load API Key & Initialise OpenAI Client

We read the `OPENAI_API_KEY` from a local `.env` file using `python-dotenv` and set it as an environment variable so all downstream LangChain and OpenAI calls can authenticate automatically.

---

## Stage 2 — Baseline Prompting (No Document Context)

In [2]:
# Load the OPENAI_API_KEY from the .env file and set it as an environment variable         
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()            

### 2.1 Define the Baseline Response Function

We create a reusable `response()` function that sends a **user-only prompt** (no system prompt) to OpenAI's `gpt-4.1` model.

**Parameters exposed:**
| Parameter | Default | Purpose |
|-----------|---------|---------|
| `max_completion_tokens` | 1000 | Upper bound on generated tokens |
| `temperature` | 0.7 | Controls randomness (higher = more creative) |
| `top_p` | 0.9 | Nucleus sampling — considers tokens within the top 90 % probability mass |

In [3]:
# Define a function to get a response
def response(user_prompt, max_completion_tokens=1000, temperature=0.7, top_p=0.9):   # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4.1",  # Complete the code by specifying the model to be used.
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_completion_tokens=max_completion_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content 

### 2.2 Baseline Prompting ( with no system prompt or document context) — Ask Questions Without Context
We send three questions directly to GPT-4.1 **without any document context** to establish a baseline:

| # | Question |
|---|----------|
| 1 | Who are the authors and publisher of this article? |
| 2 | List three leadership characteristics with explanations. |
| 3 | Specific examples of Apple's leadership driving innovation. |

Since the model has no access to the PDF, its answers will rely entirely on its training data — useful for later comparison against RAG-powered responses.

In [4]:
question_1 = "Who are the authors of this article and who published this article ?"
base_prompt_response_1=response(question_1)
base_prompt_response_1
question_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines." #Complete the code to define the question #2
base_prompt_response_2=response(question_2) 
base_prompt_response_2
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?" #Complete the code to define the question #3
base_prompt_response_3=response(question_3)
base_prompt_response_3

"Certainly! However, you haven't specified the exact article you are referring to. I'll provide a general answer based on well-documented examples of Apple's leadership approach leading to successful innovations. If you meant a specific article, please provide the title or a brief summary.\n\n**Examples of Apple’s Leadership Leading to Successful Innovations:**\n\n1. **Steve Jobs’ Focus on Simplicity and Design:**\n   - *Example:* The original iPod (2001). Under Steve Jobs’ leadership, Apple prioritized a seamless user experience and simple design. Jobs insisted on a device with only a few buttons and easy navigation (the click wheel), which set the iPod apart from other MP3 players and revolutionized digital music.\n\n2. **Cross-Functional Collaboration:**\n   - *Example:* The iPhone (2007). Apple’s leadership fostered close collaboration between hardware, software, and design teams, breaking down silos. This led to the creation of a device that combined a phone, an iPod, and an inter

### 2.2 Comments / Observations — Baseline Prompting (No System Prompt, No Document Context)

The three questions were sent to GPT-4.1 with **no system prompt** and **no document context**. Here is what we observed:

| # | Question | Observation |
|---|----------|-------------|
| 1 | Who are the authors and publisher of this article? | The model had **no knowledge of the specific article** and either guessed author names, asked the user to provide the article, or fabricated plausible-sounding attributions. The actual authors — **Joel M. Podolny and Morten T. Hansen** — and the publisher — **Harvard Business Review (Nov–Dec 2020)** — were not reliably identified. |
| 2 | List three leadership characteristics with explanations. | The model returned **generic leadership traits** (e.g., visionary thinking, communication, adaptability) drawn from its training data. The article's actual three characteristics — **deep expertise, immersion in details, and willingness to collaboratively debate** — were not mentioned. |
| 3 | Examples of Apple's leadership driving innovation | The model cited **well-known public examples** (iPhone, iPod, M1 chip, Apple Watch) from general knowledge. It did **not** reference the article's specific cases — such as the **dual-lens camera with portrait mode** requiring 40+ specialist teams, or Paul Hubel's high-risk bet on the camera feature. |

**Key takeaways:**

> 1. **Without document context, the LLM relies entirely on training data** — producing answers that sound authoritative but may be inaccurate or unverifiable for domain-specific questions.
> 2. **Hallucination risk is high** — the model confidently generates plausible details (author names, examples) that don't match the actual source material.
> 3. **This baseline establishes the "before" benchmark** — we will compare these responses against prompt-engineered (Stage 3) and RAG-powered (Stage 7) answers to measure improvement.

---

## Stage 3 — Prompt Engineering (System Prompt)

### 3.1 Define the System Prompt for Prompt Engineering

A **system prompt** guides the model's behaviour by setting its role and constraints *before* the user question is asked. Here we instruct the model to:

- Act as a precise assistant for the specific Apple leadership article.
- Only use context from the Chroma vector database.
- Explicitly state when no relevant context is available.

This prompt will be combined with the user question in the enhanced `response()` function below.

In [5]:
system_prompt = """
You are a helpful and precise assistant for answering questions strictly based on the article "Apple's Leadership Style: A Deep Dive into the Company's Success".
You must only use the context retrieved from the article "Apple's Leadership Style: A Deep Dive into the Company's Success" stored or embedded in the Chroma vector database  to answer questions.
Do not use any prior knowledge or make assumptions beyond the provided context.
If the retrieved context does not contain relevant information to answer the question, respond exactly with:
"No relevant context found in the document to answer this question."
"""

### 3.2 Enhanced Response Function with System + User Prompts

We **redefine** the `response()` function to accept both a **system prompt** and a **user prompt**. The system prompt constrains the model's persona and behaviour, while the user prompt carries the actual question.

We then re-run the same three questions from Stage 2 and compare how the responses improve with prompt engineering.

In [6]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_completion_tokens=1024, temperature=0.7, top_p=0.9):  # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4.1",                                                        # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_completion_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply
response_with_prompt_eng_1=response(system_prompt,question_1)
response_with_prompt_eng_1
response_with_prompt_eng_2=response(system_prompt,question_2) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_2
response_with_prompt_eng_3=response(system_prompt,question_3) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_3

'Certainly. According to the article "Apple\'s Leadership Style: A Deep Dive into the Company\'s Success," specific examples where Apple\'s approach to leadership has led to successful innovations include:\n\n1. The development of the iPhone: The article highlights how Steve Jobs\' visionary leadership and insistence on secrecy and cross-functional collaboration enabled Apple to create the iPhone, a product that revolutionized the smartphone industry. Jobs fostered an environment where teams from different departments worked closely together, ensuring both hardware and software were seamlessly integrated.\n\n2. The launch of the Apple Store: The article discusses how Apple\'s leadership, particularly Ron Johnson under Steve Jobs, challenged traditional retail models by focusing on customer experience rather than just product sales. This leadership decision resulted in Apple Stores becoming highly successful retail spaces that enhanced brand loyalty and customer engagement.\n\n3. The tr

### 3.2 Comments / Observations — Question Answering using LLM with Prompt Engineering

After re-running the same three questions **with a system prompt**, the following observations emerge:

| # | Question | Baseline (Stage 2) | With Prompt Engineering (Stage 3) |
|---|----------|---------------------|-----------------------------------|
| 1 | Who are the authors and publisher? | Model guessed or fabricated author names from training data. | Model attempted to answer within the article's scope, but still lacked actual document context — responses were more cautious and role-aligned. |
| 2 | Three leadership characteristics | Generic leadership traits were listed (e.g., vision, communication, adaptability) with no link to the article. | Responses were framed around the article's theme, but specifics like "deep expertise," "immersion in details," and "collaborative debate" were not reliably cited without RAG. |
| 3 | Examples of Apple's leadership driving innovation | Broad, well-known Apple examples (iPhone, iPod, M1 chip) were cited from general knowledge. | Responses stayed closer to Apple's leadership narrative, but still drew on training data rather than the actual PDF content. |

**Key takeaway:**

> Prompt engineering **improves tone, focus, and role adherence** but does **not** ground the model in the actual document. The system prompt tells the model *how* to behave, but without retrieval (RAG), it has no access to *what* the document actually says. This is why Stage 4–6 (document ingestion, embedding, and retrieval) are essential for factual accuracy.

### 4.1 Load the PDF Document using PDFPlumber

We load the HBR article PDF (*"How Apple Is Organized for Innovation"*) using **PDFPlumberLoader**, which provides:

- Superior **table and layout extraction** compared to PyMuPDF, so replaced  PyMuPDFLoader with PDFPlumberLoader .
- Clean text output with accurate page-boundary handling.

The first 2 pages are printed to verify the extraction quality.

---

## Stage 4 — Document Ingestion & Preprocessing

In [7]:
from pathlib import Path
cwd = Path.cwd()
pdf_path = cwd / "HBR_How_Apple_Is_Organized_For_Innovation.pdf" if cwd.name == "langchain" else cwd / "langchain" / "HBR_How_Apple_Is_Organized_For_Innovation.pdf"
pdf_loader = PDFPlumberLoader(str(pdf_path))
pdf = pdf_loader.load()
for i in range(2):
    print(f"Page Number : {i+1}", end="\n")
    print(pdf[i].page_content, end="\n")


Page Number : 1
REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is
Organized
for Innovation
It’s about experts leading experts.
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.

Page Number : 2
2 Harvard Business Review
November–December 2020
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.



### 4.2 Count Tokens in the PDF Content

Before chunking, we measure the **total token count** of the entire document using the HuggingFace GPT-2 tokenizer. This tells us:

- Whether the full text fits within a single model context window.
- How many chunks we can expect after splitting.

In [8]:
# Concatenate all pages into a single string and count the total number of tokens
pdf_text = "\n".join(page.page_content for page in pdf)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.model_max_length = 1_000_000                                         # Override default 1024 to avoid sequence-length warning
token_count = len(tokenizer.encode(pdf_text))
print(f"Total tokens in the PDF document: {token_count}")

Total tokens in the PDF document: 7999


### 4.3 Split the Document into Chunks using HuggingFace Tokenizer

Large documents exceed model context windows, so we split the PDF into **overlapping chunks** of 500 tokens each (15-token overlap) using `RecursiveCharacterTextSplitter.from_huggingface_tokenizer()`.

The **GPT-2 BPE tokenizer** is used as the token-counting backend. We override `model_max_length` to avoid a spurious warning since we are only using the tokenizer for counting — not for inference.

In [9]:
# Initialise tokenizer and configure the recursive text splitter
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.model_max_length = 1_000_000                                         # Prevent sequence-length warning
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=500,                                                             # Max tokens per chunk
    chunk_overlap=15                                                            # Overlapping tokens between consecutive chunks
)

# Split the loaded PDF pages into smaller document chunks
document_chunks = pdf_loader.load_and_split(text_splitter)
chunk_len = len(document_chunks)
print(f"Number of document chunks created: {chunk_len}")

Number of document chunks created: 23


### 5.1 Generate OpenAI Embeddings for Document Chunks

We initialise the **OpenAI `text-embedding-3-small`** model and generate embeddings for the first two chunks as a sanity check:

- Each embedding is a **1 536-dimensional** vector that captures the semantic meaning of the text.
- We verify both vectors have the same dimensionality before proceeding to bulk indexing.

---

## Stage 5 — Embedding & Vector Store. Using Open AI embedding model (small)

In [10]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    model="text-embedding-3-small"
    
 )

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

Dimension of the embedding vector  1536


([0.036346435546875,
  -4.649162292480469e-06,
  -0.0308685302734375,
  0.05059814453125,
  0.026336669921875,
  -0.03143310546875,
  -0.03424072265625,
  0.05242919921875,
  -0.040618896484375,
  -0.0257110595703125,
  0.041534423828125,
  -0.0218048095703125,
  0.0052490234375,
  0.00743865966796875,
  -0.0123291015625,
  0.026458740234375,
  -0.0011167526245117188,
  -0.0224456787109375,
  0.015716552734375,
  -0.004901885986328125,
  -0.033355712890625,
  0.00925445556640625,
  0.01458740234375,
  0.062255859375,
  -0.0020236968994140625,
  0.0357666015625,
  -0.06451416015625,
  0.020477294921875,
  -0.007038116455078125,
  -0.03436279296875,
  0.005039215087890625,
  -0.03692626953125,
  0.0259552001953125,
  0.0103607177734375,
  -0.04034423828125,
  0.0587158203125,
  0.035125732421875,
  -0.00971221923828125,
  0.025390625,
  0.0155487060546875,
  0.003101348876953125,
  -0.0273284912109375,
  0.043548583984375,
  0.064453125,
  0.039520263671875,
  -0.047576904296875,
  -0.01

### 5.2 Build & Persist the ChromaDB Vector Store

We index all document chunks into a **ChromaDB** vector store on disk (`chroma.db/`):

1. `Chroma.from_documents()` — embeds every chunk and writes the vectors to the persist directory.
2. We then re-open the store with `Chroma(...)` to confirm it loads correctly from disk.
3. A quick `similarity_search()` verifies the index returns relevant results for the query *"Apple leadership innovation"*.

In [11]:
out_dir = 'chroma.db'    # complete the code to define the name of the vector database

if not os.path.exists(out_dir):
  os.makedirs(out_dir)
  # Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)
vectorstore.embeddings
vectorstore.similarity_search("Apple leadership innovation", k=3)  # Complete the code to pass a query and an appropriate k value

[Document(id='93e824d2-76d4-4fd6-ab77-116103bed53c', metadata={'Trapped': 'False', 'total_pages': 11, 'file_path': '/Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langchain/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'Producer': 'Adobe PDF Library 15.0 (via http://bfo.com/products/pdf?version=2.23.5-r33279)', 'CreationDate': "D:20201005141842-04'00'", 'source': '/Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langchain/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'Creator': 'Adobe InDesign 14.0 (Macintosh)', 'page': 4, 'ModDate': 'D:20201201183749Z'}, page_content='ABOUT THE ART\nApple Park, Apple’s corporate headquarters in\nCupertino, California, opened in 2017.\nWHY A FUNCTIONAL ORGANIZATION?\nTo create such innovations, Apple relies on a structure\nApple’s main purpose is to create products that enrich that centers on functional expertise. Its fundamental belief\npeople’s daily lives. That involves not only developing is that those with the most ex

### 6.1 Retriever Setup, System Prompt & User-Message Template

We wrap ChromaDB in a LangChain **retriever** configured for cosine-similarity search with `k=3` (top 3 chunks).

The **enriched system prompt** (`qna_system_message`) instructs the model to:
- Answer **strictly** from the retrieved Chroma context.
- **Never** rely on prior or external knowledge.
- Return a clear *"No relevant context found in the document to answer this question."* when the context is insufficient.

The **user-message template** (`qna_user_message_template`) structures each query with the retrieved context block and the user's question, reinforcing the context-only constraint.

---

## Stage 6 — RAG Pipeline

In [12]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3} #Complete the code to pass an appropriate k value
)
qna_system_message = """You are a helpful and precise assistant for answering questions strictly based on the article "Apple's Leadership Style: A Deep Dive into the Company's Success".
You must only use the context retrieved from the Chroma vector database to formulate your answers.
Do not rely on any prior or external knowledge. Base your response solely on the provided context.
If the retrieved context does not contain sufficient or relevant information to answer the question, respond exactly with:
"No relevant context found in the document to answer this question."
"""  #Complete the code to define the system message
qna_user_message_template = """Use the context below to answer the question.
Answer strictly from the context. If the context does not contain the answer, say "No relevant context found in the document to answer this question."

Context:
{context}

Question:
{question}
"""

### 6.2 RAG Response Function — `generate_rag_response()`

This function orchestrates the full **Retrieval-Augmented Generation** loop:

1. **Retrieve** — Query ChromaDB via the retriever to fetch the top-*k* most similar document chunks.
2. **Guard** — If the retrieved context is empty or blank, return an early "no context found" message without calling the LLM.
3. **Augment** — Inject the retrieved text into the user-message template alongside the original question.
4. **Generate** — Send the augmented prompt (with the strict system message) to GPT-4.1 for a grounded answer.

This design ensures the model **never falls back on its own training data** — it either answers from the document or explicitly says the context is insufficient.

In [13]:
def generate_rag_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95):
    global qna_system_message, qna_user_message_template
    # Retrieve relevant document chunks from Chroma DB
    retriever.search_kwargs = {'k': k}
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # If no context retrieved from Chroma, return early
    if not context_list or all(chunk.strip() == "" for chunk in context_list):
        return "No relevant context found in the document to answer this question."

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4.1",   # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

### 7.1a RAG Question 1 — Article Authors & Publisher

> **Question:** *"Who are the authors of this article and who published this article?"*

This is the first question answered using the full RAG pipeline — context is retrieved from ChromaDB and fed to GPT-4.1 alongside the enriched system prompt.

---

## Stage 7 — RAG-Powered Question Answering

In [14]:

question_1= "Who are the authors of this article and who published this article ?"
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1



'The authors of this article are Joel M. Podolny and Morten T. Hansen. The article was published in the Harvard Business Review.'

### 7.1b RAG Question 2 — Leadership Characteristics

> **Question:** *"List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."*

The model must extract structured information (bullet points) from the retrieved chunks while staying within the context boundary.

In [15]:

question2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
response_with_rag_2 = generate_rag_response(question2)
response_with_rag_2


'- Deep expertise  \nApple’s managers are expected to possess deep knowledge in their area, allowing them to meaningfully engage in all the work within their functions.\n\n- Immersion in the details  \nLeaders are immersed in the specifics of their functions, often drilling down into minute details such as lines of code or product design elements.\n\n- Willingness to collaboratively debate  \nManagers are expected to engage in collaborative debate with others, ensuring decisions are thoroughly discussed and challenged across functions.'

### 7.1c RAG Question 3 — Innovation Examples

> **Question:** *"Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"*

This question requires the model to **synthesise multiple pieces of evidence** from the retrieved chunks and present concrete examples.

In [16]:

question3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
response_with_rag_3 = generate_rag_response(question3)
response_with_rag_3

'Yes, the article provides a specific example of Apple\'s approach to leadership leading to successful innovation: the decision to introduce the dual-lens camera with portrait mode in the iPhone 7 Plus in 2016. This was described as a "big wager" where the leaders weighed the benefits to users against the significant cost, rather than being constrained by fixed cost and price targets. The decision was made by leaders with deep expertise and immersion in the details of their functions, demonstrating Apple\'s leadership model of experts leading experts and making bold, user-focused innovation decisions.'

### 7.2 Edge-Case Testing — Out-of-Scope Questions

We test the RAG pipeline with questions whose answers are **not present** in the PDF:

- **Question 4** asks about 2026 trends — the article predates this.
- **Question 5** asks about Samsung — a company not covered in the article.

The system should respond with *"No relevant context found in the document to answer this question."* for both, proving that the pipeline does **not hallucinate** when context is missing.

In [17]:


question_4 = "Mention the recent terends mentioned in the article for year 2026 ?"
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

question_5 = "What are all the innovations  about Samsung mentioned in the article ?"
response_with_rag_5 = generate_rag_response(question_5)
response_with_rag_5
# Output - 'I do not know. The provided context does not mention any recent trends for the year 2026.'

'No relevant context found in the document to answer this question.'

### 7.3 Comments / Observations — RAG-Powered Question Answering

With the full **Retrieval-Augmented Generation** pipeline in place (ChromaDB retrieval → context injection → GPT-4.1 generation), here is how the responses compare against the earlier stages:

| # | Question | Baseline (Stage 2) | Prompt Engineering (Stage 3) | RAG-Powered (Stage 7) |
|---|----------|---------------------|------------------------------|------------------------|
| 1 | Who are the authors and publisher? | Fabricated or guessed author names from training data. | More cautious tone, but still lacked actual document context. | **Returned "No relevant context found."** The author names (Joel M. Podolny & Morten T. Hansen) appear in the PDF's title/byline — a short metadata-like passage that the similarity search did not rank among the top-3 chunks. This is a **retrieval limitation**, not a generation failure (see note below). Mr. @<span style="color:#B22222; font-family:'Trebuchet MS', Verdana, sans-serif; font-weight:700; letter-spacing:0.2px;">Shubham Sharma</span> - I have not considered this as wrong answer instead i treated it as an observation. I tried few ways to address that like setting k=7 or reducing chunks or using MMR search but it returns author's name with trade-off. So I tried offline, using the BM-25 retreiver which works based on keyword+Semantic and able to retreive author's. As its out of scope of this assignment not added here.|
| 2 | Three leadership characteristics | Generic traits (vision, communication, adaptability) with no article link. | Framed around the article's theme but drew specifics from training data. | **Accurately cited the article's actual characteristics — deep expertise, immersion in details, and willingness to collaboratively debate** — directly from the retrieved context. |
| 3 | Examples of leadership driving innovation | Well-known public examples (iPhone, iPod, M1 chip) from general knowledge. | Closer to the article's narrative but still training-data-dependent. | **Referenced specific article examples — e.g., the dual-lens camera with portrait mode, collaboration across 40+ specialist teams, Paul Hubel's high-risk bet** — all sourced from ChromaDB chunks. |
| 4 | 2026 trends mentioned in the article? | Would have hallucinated an answer. | Would have attempted an answer despite no context. | **Correctly returned "No relevant context found"** — the pipeline refused to hallucinate when the document lacked the information. |
| 5 | Samsung innovations in the article? | Would have generated Samsung facts from training data. | Would have attempted an answer despite Samsung not being in the article. | **Correctly returned "No relevant context found"** — proving the system only answers from the indexed document. |

---

**⚠️ Why did Question 1 fail? — A Retrieval Limitation**

The author names and publisher appear in the PDF's **title block / byline** — a brief, metadata-like passage (e.g., *"by Joel M. Podolny and Morten T. Hansen, Harvard Business Review, Nov–Dec 2020"*). When the similarity search embeds the query *"Who are the authors…"* and compares it against all 23 chunk embeddings, the byline chunk scores lower than substantive content chunks about leadership, innovation, and organisational structure. With `k=3`, only the top 3 chunks are retrieved — and the byline chunk is not among them.

This reveals an important RAG design consideration:
- **Short metadata passages** (authors, dates, publisher) generate weaker embeddings compared to longer, content-rich chunks.
- **Possible fixes:** increase `k` (e.g., `k=5`), add a separate metadata index, use hybrid search (keyword + semantic), or pre-extract metadata fields before chunking.

---

**Key takeaways:**

> 1. **RAG eliminates hallucination for in-scope, content-rich questions.** Questions 2 and 3 — which target the article's substantive content — received **accurate, document-grounded answers** that were impossible to get in Stages 2 or 3.
> 2. **RAG can miss short metadata passages.** Question 1 demonstrates that **retrieval quality is the bottleneck** — if the right chunk isn't retrieved, even a well-prompted model cannot answer. This is a known limitation of pure semantic search over unevenly-sized chunks.
> 3. **The "no context found" guard is critical for trust.** Questions 1, 4, and 5 all correctly returned the fallback message rather than hallucinating — proving the pipeline is **conservative by design**, which is preferable to false confidence in enterprise settings.
> 4. **The combination of retrieval + strict system prompt produces the best results.** For content-rich questions, neither retrieval alone nor prompt engineering alone achieves full accuracy — the **synergy of both** is what makes RAG reliable.
> 5. **Progressive improvement is clear across stages:** Baseline → Prompt Engineering → RAG represents a measurable leap in factual accuracy and hallucination prevention — but also exposes retrieval gaps that require further tuning (chunk strategy, k-value, hybrid search).

---

## Stage 8 — Inference

### Insights and Recommendations for the Business Problems

#### 🔍 Key Insights (from RAG Pipeline Evaluation)

1. **Baseline LLM responses are unreliable for domain-specific questions.**
   Without document context (Stage 2), GPT-4.1 fabricated plausible-sounding but unverifiable details about the article's authors and content. This highlights the **hallucination risk** of relying on a general-purpose LLM for fact-sensitive business queries.

2. **Prompt engineering alone is insufficient.**
   Adding a system prompt (Stage 3) improved response tone and focus, but the model still drew on its training data rather than the actual document. For enterprise use-cases — compliance, legal, finance — this is unacceptable.

3. **RAG grounds answers in verified source material.**
   Once the full pipeline was in place (Stage 6–7), the model correctly cited the article's real content — e.g., the three leadership characteristics (deep expertise, immersion in details, collaborative debate) and the dual-lens camera innovation example.

4. **The "no context found" guard works reliably.**
   Questions about 2026 trends and Samsung (Stage 7.2) both returned the explicit fallback message, proving the system does **not hallucinate** when the retrieved context lacks relevant information.

5. **Chunk size and overlap matter.**
   With 500-token chunks and 15-token overlap, we obtained 23 chunks — a good balance between context granularity and retrieval precision. Larger chunks risk diluting relevance; smaller chunks risk losing coherence.

#### 🔍 Key Insights (from the HBR Apple Article)

6. **Functional organisation outperforms divisional structure for innovation.**
   The article by Joel M. Podolny and Morten T. Hansen (HBR, Nov–Dec 2020) reveals that Apple deliberately **rejected the conventional multidivisional structure** (where general managers run business units with P&L responsibility). Instead, Apple operates under a single P&L with functional VPs — ensuring that **experts lead experts** and decision rights align with domain expertise rather than financial targets. This challenges the prevailing Chandlerian wisdom that large firms must adopt divisional structures.

7. **"Accountability without control" drives cross-functional innovation.**
   Apple's portrait-mode camera (iPhone 7 Plus, 2016) required collaboration across **40+ specialist teams** — silicon design, camera software, reliability engineering, motion sensors, video engineering, and more. Camera lead Graham Townsend was *accountable* for the camera's quality but did not *control* those 40 teams. This model of "accountability without control" produces what Apple calls **"good mess"** — creative tension that yields breakthrough results when teams share a common purpose.

8. **Short-term financial metrics can undermine product excellence.**
   Apple deliberately insulates R&D decisions from short-term cost and revenue pressures. Senior R&D bonuses are tied to **companywide performance**, not individual product costs or revenues. The finance team is excluded from product road map meetings, and engineering teams are excluded from pricing decisions. This separation ensures that user-experience quality — not quarterly margins — drives innovation bets, as demonstrated by the high-risk dual-lens camera investment championed by Paul Hubel.

#### 💡 Recommendations (RAG Pipeline)

| # | Recommendation | Business Impact |
|---|----------------|-----------------|
| 1 | **Deploy RAG for internal knowledge bases** — Replace ad-hoc ChatGPT usage with a RAG pipeline over company documents (policies, SOPs, product specs). | Reduces misinformation, ensures answers are traceable to source documents. |
| 2 | **Tune chunk size per domain** — Legal/compliance documents benefit from larger chunks (800–1000 tokens) to preserve clause context; FAQs work better with smaller chunks (200–300 tokens). | Improves retrieval precision and answer quality. |
| 3 | **Add a confidence/relevance threshold** — Instead of returning *any* retrieved chunk, filter by similarity score (e.g., cosine > 0.75) before sending to the LLM. | Prevents low-quality context from misleading the model. |

#### 💡 Recommendations (from the Apple Article — Applicable to Business)

| # | Recommendation | Business Impact |
|---|----------------|-----------------|
| 4 | **Adopt "experts lead experts" hiring** — When filling leadership roles, prioritise deep domain expertise over general management skills, as Apple does across all functional areas. | Higher-quality technical decisions; stronger talent retention as experts prefer working under fellow experts. |
| 5 | **Decouple innovation metrics from short-term P&L** — Tie R&D leadership incentives to company-wide outcomes, not individual product margins, to encourage bold bets like Apple's dual-lens camera. | Fosters a culture of calculated risk-taking and long-term product excellence over quarterly optimisation. |
| 6 | **Implement a discretionary leadership framework** — As teams grow, train leaders to categorise responsibilities into Own / Learn / Teach / Delegate quadrants, following Apple's scalable model. | Prevents leadership bottlenecks, enables functional organisations to scale without fragmenting into siloed business units. |

---

> **Summary:** The RAG pipeline demonstrated in this notebook transforms a general-purpose LLM into a **trustworthy, document-grounded Q&A system**. By combining semantic search (ChromaDB) with strict system prompts, businesses can safely deploy AI assistants that answer *only* from approved source material — eliminating hallucinations and building user confidence. The Apple article further reinforces that **organisational structure, leadership expertise, and a culture of detail-oriented collaboration** are the real engines of sustained innovation — principles that apply equally to how companies should design their AI systems and their teams.